# Lecture 3 — Model Answer
## Line Charts & Slopegraphs: CO2 Emissions

---


In [ ]:
import pandas as pd
import plotly.express as px


# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


## Task 1 — Model Answer


In [ ]:


asian = df[df['Region'] == 'Asia'].copy()
highlight = 'China'

# setting China in a punchy color and all other countries in grey
colour_map = {country: ('red' if country == highlight else '#DDDDDD') for country in asian['Country'].unique()}

# core px figure
fig = px.line(
    asian.sort_values('Year'),
    x='Year', y='CO2_Mt',
    color='Country',
    color_discrete_map=colour_map,
    title="China's CO2 emissions are 4x any other Asian country — and still rising",
    labels={'CO2_Mt': 'CO2 Emissions (Mt)', 'Year': ''})

# Update each trace individually — width depends on whether it's the highlight
for trace in fig.data:
    trace.showlegend = False
    trace.line.width = 3 if trace.name == highlight else 1

# calculating the x and y positions of the annotation replacing the legend
last = asian[(asian['Country'] == highlight) & (asian['Year'] == asian['Year'].max())]

fig.add_annotation(
    x=last['Year'].values[0], y=last['CO2_Mt'].values[0],
    text=f'<b>{highlight}</b>', showarrow=False, 
    xanchor='left', xshift=8, # left anchoring + shifting by 8 pixels to avoid overlapping with line
    font=dict(color='red', size=12, family='Arial')
)

fig.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    yaxis=dict(gridcolor='#EEEEEE'),
    xaxis=dict(showgrid=False),
    margin=dict(l=60, r=100, t=55, b=40),
    height=500, width=1000
)

fig.show()


## Task 2 — Model Answer


In [ ]:
# using a slopegraph is difficult here because of the very wide variations in the magnitude of change 
# Logging the Y-axis here can be a solution in some cases to dampen the visual effect of the very wide variations

# prep data
reg = (df[df['Year'].isin([2000, 2022])]
       .groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index())
changes = reg.groupby('Region')['CO2_Mt'].agg(['first', 'last'])
colour_map = {region: 'red' if row['last'] > row['first'] else 'green' for region, row in changes.iterrows()}

fig = px.line(
    reg.sort_values('Year'),
    x='Year', y='CO2_Mt',
    color='Region',
    color_discrete_map=colour_map,
    markers=True,
    log_y=True,
    labels={'CO2_Mt': '', 'Year': ''}
)

for region in changes.index:
    d = reg[reg['Region'] == region].sort_values('Year')
    fig.update_traces(
        selector=dict(name=region),
        mode='lines+markers+text',
        text=[f'{d["CO2_Mt"].iloc[0]:.0f}', f'{region}<br>{d["CO2_Mt"].iloc[1]:.0f}'],
        textposition=['middle left', 'middle right'],
        textfont=dict(size=10, color=colour_map[region], family='Arial'),
        showlegend=False
    )

fig.update_layout(
    title='Asia and Latin America drove global CO2 growth — Europe and North America reversed course',
    xaxis=dict(tickvals=[2000, 2022], ticktext=['2000', '2022'],
               showgrid=False, range=[1993, 2030]),
    yaxis=dict(showgrid=False, showticklabels=False, title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=160, r=160, t=55, b=40), 
    height=600, width=1000
)

fig.show()

In [ ]:
# another solution would be to calculate the percentages of change and do a bar chart 


# add the pct change to the prepped data
changes['pct_change'] = ((changes['last'] - changes['first']) / changes['first'] * 100).round(1)
colour_map = {region: 'green' if row['pct_change'] > 0 else 'red' for region, row in changes.iterrows()}

fig = px.bar(
    changes.reset_index().sort_values('pct_change'),
    x='pct_change', y='Region',
    orientation='h',
    color='Region',
    color_discrete_map=colour_map,
    text='pct_change',
    labels={'pct_change': 'Change in CO2 Emissions (%)', 'Region': ''},
    title='From 2000 to 2022: Asia and Latin America drove global CO2 growth — Europe and North America reversed course'
)

fig.update_traces(
    showlegend=False,
    texttemplate='%{text:+.1f}%',
    textposition='outside'
)

fig.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    xaxis=dict(showgrid=False, zeroline=True, zerolinecolor='#AAAAAA', zerolinewidth=1),
    yaxis=dict(showgrid=False),
    margin=dict(l=120, r=80, t=55, b=40),
    height=400
)

fig.show()